# 14.2 · 时间序列分解 / Time Series Decomposition

> **课程定位 / Where this fits**
> 第 2 课，**Part 14 · 时间序列**。把序列"拆开看"。
> Lesson 2, **Part 14 · Time Series**. Taking a series apart to see inside.
>
> 14.1 看到序列里同时混着趋势、季节、噪声。**分解(decomposition)** 就是把它们**显式拆成三部分**:**趋势(trend) + 季节(seasonal) + 残差(residual)**。拆开后, 每部分的形状一目了然——趋势告诉你长期走向, 季节告诉你周期规律, 残差是去掉这两者后剩下的"意外"(异常检测就盯着它)。本课讲清**加法 vs 乘法**分解的区别、何时用哪种, 并用经典分解和更稳健的 **STL** 分解航空客运量。
> Lesson 14.1 showed trend, seasonality, and noise mixed together. **Decomposition** explicitly **splits a series into three parts**: **trend + seasonal + residual**. Once split, each part's shape is clear — the trend shows the long-term direction, the seasonal shows the periodic pattern, and the residual is the "surprise" left over (anomaly detection watches this). We cover **additive vs multiplicative** decomposition, when to use each, and decompose the airline data with classical and the more robust **STL**.
>
> 💼 **实战/面试视角**："加法vs乘法分解怎么选 / STL 比经典分解好在哪 / 分解的用途(季节调整/异常)" 是时序分析常考。
> 💼 **Practical/interview angle:** "additive vs multiplicative / why STL over classical / uses (seasonal adjustment/anomaly)" — common.

> 📐 **符号约定 / Notation**
> - 加法 $y_t = T_t + S_t + R_t$ / additive
> - 乘法 $y_t = T_t \times S_t \times R_t$ / multiplicative

> 💡 **面试相关 / Interview-relevant**
> - "时间序列分解成哪三部分"（出镜率 ★★★★）
> - "加法vs乘法分解, 怎么判断用哪个"（★★★★★）
> - "STL 分解相比经典分解的优势"（★★★★）
> - "分解后残差有什么用"（★★★，异常检测/建模）

---

## 学习目标 / Learning Objectives
1. 理解分解 = 趋势 + 季节 + 残差。
   Understand decomposition = trend + seasonal + residual.
2. 区分**加法 vs 乘法**分解, 会判断用哪种。
   Distinguish additive vs multiplicative; decide which to use.
3. 用经典分解和 **STL** 分解, 对比效果。
   Decompose with classical and STL methods; compare.
4. 知道分解的实战用途(季节调整/异常检测)。
   Know practical uses (seasonal adjustment/anomaly detection).

## 目录 / TOC
1. [分解:趋势+季节+残差 ⭐](#1)
2. [加法 vs 乘法分解 ⭐](#2)
3. [STL 分解 ⭐](#3)
4. [分解的用途 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 分解:趋势+季节+残差 ⭐ / Decomposition: Trend + Seasonal + Residual

**分解**把观测序列 $y_t$ 拆成三个可解释的成分：
**Decomposition** splits the observed series $y_t$ into three interpretable parts:
- **趋势 $T_t$**:长期的、平滑的走向(通常用移动平均估计)。
  **Trend $T_t$:** the long-term, smooth direction (often estimated by a moving average).
- **季节 $S_t$**:固定周期内重复的模式(月度数据周期=12)。
  **Seasonal $S_t$:** the repeating within-period pattern (period 12 for monthly data).
- **残差 $R_t$**:去掉趋势和季节后剩下的随机部分。
  **Residual $R_t$:** the random leftover after removing trend and seasonal.

拆开的价值:**分而治之**——分别理解长期走向、周期规律、和"异常波动";也是预测和异常检测的基础。先做经典分解看看三部分长什么样。
The value: **divide and conquer** — understand the long-term trend, periodic pattern, and "anomalous wiggles" separately; also the basis for forecasting and anomaly detection. Let's run a classical decomposition.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")
from statsmodels.tsa.seasonal import seasonal_decompose, STL
sns.set_theme(style="whitegrid")
ap = [112,118,132,129,121,135,148,148,136,119,104,118, 115,126,141,135,125,149,170,170,158,133,114,140,
      145,150,178,163,172,178,199,199,184,162,146,166, 171,180,193,181,183,218,230,242,209,191,172,194,
      196,196,236,235,229,243,264,272,237,211,180,201, 204,188,235,227,234,264,302,293,259,229,203,229,
      242,233,267,269,270,315,364,347,312,274,237,278, 284,277,317,313,318,374,413,405,355,306,271,306,
      315,301,356,348,355,422,465,467,404,347,305,336, 340,318,362,348,363,435,491,505,404,359,310,337,
      360,342,406,396,420,472,548,559,463,407,362,405, 417,391,419,461,472,535,622,606,508,461,390,432]
idx = pd.date_range("1949-01", periods=len(ap), freq="MS"); ts = pd.Series(ap, index=idx, name="passengers")
print(f"AirPassengers: {len(ts)} 个月度观测; 周期=12(月度年度季节)")

# 经典分解(乘法型, 因为季节幅度随水平增大) / classical decomposition (multiplicative)
result = seasonal_decompose(ts, model="multiplicative", period=12)
fig = result.plot(); fig.set_size_inches(11, 8)
fig.suptitle("经典分解(乘法): 观测 = 趋势 × 季节 × 残差", y=1.0)
plt.tight_layout(); plt.show()
print("自上而下: 原序列 / 趋势(平滑上升) / 季节(每年重复的固定形状) / 残差(剩余随机波动)")
print("分解把混在一起的成分显式拆开 → 各自规律一目了然")


<a id="2"></a>
## 2. 加法 vs 乘法分解 ⭐ / Additive vs Multiplicative

分解有两种模型, **怎么选是面试常考点**：
Two models; **choosing is a common interview point:**
- **加法 $y_t = T_t + S_t + R_t$**:季节波动的**幅度大致恒定**(不管整体水平多高, 每年夏天都多卖 100 件)。适合波动幅度稳定的序列。
  **Additive $y_t = T_t + S_t + R_t$:** seasonal swings have **roughly constant amplitude** (always +100 in summer regardless of level). For series with stable variation.
- **乘法 $y_t = T_t \times S_t \times R_t$**:季节波动的**幅度随整体水平成比例增大**(水平翻倍, 夏季波峰也翻倍)。航空客运量就是典型——后期波峰明显比早期大。
  **Multiplicative $y_t = T_t \times S_t \times R_t$:** seasonal amplitude **grows proportionally with the level** (double the level → double the swing). The airline data is classic — later peaks are much bigger than early ones.

**怎么判断**:看图——**波动幅度是否随水平增大**? 增大→乘法; 恒定→加法。或者: 对乘法型**取对数**就变成加法型($\log(T\cdot S\cdot R)=\log T+\log S+\log R$), 这也是 14.1 取对数的原因。
**How to judge:** look — does the **amplitude grow with the level**? Growing → multiplicative; constant → additive. Also: taking the **log** of a multiplicative series makes it additive ($\log(T\cdot S\cdot R)=\log T+\log S+\log R$), the reason we logged in 14.1.

下面对同一数据做两种分解, 用**残差是否"干净"(更随机、更接近常数)** 来判断哪个模型更合适。
Below we run both on the same data and judge by **which residual is "cleaner" (more random, flatter)**.


In [ ]:
add = seasonal_decompose(ts, model="additive", period=12)
mul = seasonal_decompose(ts, model="multiplicative", period=12)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(add.resid); axes[0].axhline(0, color="gray", ls="--"); axes[0].set_title("加法分解的残差: 早期小、后期大(有结构残留)")
axes[1].plot(mul.resid); axes[1].axhline(1, color="gray", ls="--"); axes[1].set_title("乘法分解的残差: 全程围绕1更均匀(更干净)")
plt.tight_layout(); plt.show()
# 用残差的"非恒定性"量化: 加法残差的标准差是否随时间变 / quantify: does residual std change over time?
add_resid = add.resid.dropna(); mul_resid = mul.resid.dropna()
add_early, add_late = add_resid[:60].std(), add_resid[-60:].std()
print(f"加法残差标准差: 前期 {add_early:.1f} vs 后期 {add_late:.1f}  (相差大 → 加法没拟合好幅度增长)")
print(f"乘法残差: 围绕 1 波动, 前期std {mul_resid[:60].std():.3f} vs 后期 {mul_resid[-60:].std():.3f} (更一致)")
print("\n判断: AirPassengers 季节幅度随水平增大 → 乘法分解更合适(残差更均匀干净)")
print("经验法则: 波动随水平增大→乘法(或先取log); 波动恒定→加法")


<a id="3"></a>
## 3. STL 分解 ⭐ / STL Decomposition

上面的经典分解有局限(面试点): ①季节形状被假设**每年完全相同**(用固定的季节均值);②对**异常值敏感**;③趋势用简单移动平均, 端点会丢数据。
The classical decomposition has limits (interview): ① the seasonal shape is assumed **identical every year** (fixed seasonal means); ② it's **sensitive to outliers**; ③ the trend uses a simple moving average, losing endpoints.

**STL(Seasonal-Trend decomposition using Loess)** 是更稳健的现代方法:用**局部加权回归(Loess)** 平滑地估计趋势和季节, 因此能①**允许季节形状随时间缓慢变化**;②对异常值**更稳健**;③更灵活(可调趋势/季节平滑程度)。是实务首选之一。
**STL (Seasonal-Trend decomposition using Loess)** is a robust modern method: it uses **locally weighted regression (Loess)** to smoothly estimate trend and seasonal, so it ① **lets the seasonal shape slowly evolve over time**; ② is **more robust to outliers**; ③ is more flexible. A practical favorite.

> 注: STL 本身是**加法**分解。对乘法型数据(如本例), 标准做法是先 `log`, STL 完再 `exp` 回去。
> Note: STL is **additive**. For multiplicative data, the standard trick is to STL on the `log` then `exp` back.


In [ ]:
stl = STL(np.log(ts), period=12, robust=True).fit()      # 对 log 做 STL(robust=对异常稳健) / STL on log, robust
fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
axes[0].plot(ts); axes[0].set_title("观测序列")
axes[1].plot(np.exp(stl.trend)); axes[1].set_title("STL 趋势(平滑上升)")
axes[2].plot(stl.seasonal); axes[2].set_title("STL 季节(log尺度; 注意形状可随年份缓慢变化)")
axes[3].plot(stl.resid); axes[3].axhline(0, color="gray", ls="--"); axes[3].set_title("STL 残差(去趋势去季节后的随机部分)")
plt.tight_layout(); plt.show()
print("STL 用 Loess 平滑估计趋势/季节, 允许季节缓慢演变 + 对异常稳健 → 比经典分解更灵活实用")
print("STL是加法分解; 乘法型数据先取log再STL(完后exp回去), 本例就是这么做的")


<a id="4"></a>
## 4. 分解的用途 + 小结 ⭐ / Uses of Decomposition

分解不只是"看清结构", 还有实战用途(面试可举)：
Decomposition isn't just "seeing structure" — it has practical uses:
- **季节调整(seasonal adjustment)**:从序列里**去掉季节成分**(deseasonalize), 看清真实的趋势走向。新闻里的"经季节调整后的失业率/GDP"就是这么算的——剔除"每年都有的规律波动", 才能判断真实变化。
  **Seasonal adjustment:** **remove the seasonal component** (deseasonalize) to see the true trend. The "seasonally adjusted unemployment/GDP" in the news works this way — strip the regular yearly swing to judge real change.
- **简单预测**:分别预测趋势(外推)和季节(重复), 再组合——一个朴素但有效的基线。
  **Simple forecasting:** forecast trend (extrapolate) and seasonal (repeat) separately, then combine — a naive but effective baseline.
- **异常检测**:盯着**残差**——正常情况下残差应是小的随机波动; 残差突然变得很大 = 异常(14.10 详讲)。
  **Anomaly detection:** watch the **residual** — normally small random noise; a sudden large residual = anomaly (detailed in 14.10).

下面演示季节调整(去季节)。
Below we demonstrate seasonal adjustment.


In [ ]:
# 季节调整: 原序列 ÷ 季节成分(乘法型) → 去季节, 露出纯趋势 / seasonal adjustment (multiplicative): divide out seasonal
seasonal_adjusted = ts / mul.seasonal
fig, ax = plt.subplots(figsize=(11, 4))
ts.plot(ax=ax, label="原始(含季节波动)", alpha=0.5)
seasonal_adjusted.plot(ax=ax, label="季节调整后(去掉季节)", color="red", lw=2)
ax.legend(); ax.set_title("季节调整: 去掉'每年都有的规律波动', 露出更平滑的真实趋势(类似'经季节调整的GDP')")
plt.tight_layout(); plt.show()
print("去季节后曲线平滑很多 → 能看清真实长期走向, 不被季节波动干扰")
print("分解用途: 季节调整(看真实趋势) / 朴素预测(趋势外推+季节重复) / 异常检测(盯残差)")


```
分解: 把序列拆成 趋势T + 季节S + 残差R; 分而治之, 各成分形状一目了然
加法 y=T+S+R: 季节幅度恒定; 乘法 y=T×S×R: 季节幅度随水平成比例增大(取log→变加法)
怎么选: 波动随水平增大→乘法(AirPassengers典型); 恒定→加法; 看残差哪个更干净
经典分解局限: 季节假设每年相同 + 对异常敏感 + 端点丢数据
STL: Loess平滑分解, 允许季节缓慢演变 + 对异常稳健 + 更灵活; 是加法(乘法数据先log)
用途: 季节调整(去季节看真实趋势) / 朴素预测 / 异常检测(盯残差)
```

### 💡 面试速查 / Interview cheat-sheet
1. **三成分**: 趋势+季节+残差; 加法(y=T+S+R)或乘法(y=T×S×R)。
   Three parts: trend+seasonal+residual; additive or multiplicative.
2. **加法vs乘法**: 季节幅度恒定→加法; 随水平增大→乘法(或先log)。
   Additive vs multiplicative: constant swing → additive; growing with level → multiplicative.
3. **STL**: Loess稳健分解, 季节可演变+抗异常+灵活; 优于经典分解。
   STL: robust Loess decomposition, evolving seasonality + outlier-robust; beats classical.
4. **判断方法**: 看波动是否随水平增大 / 比较残差哪个更干净。
   How to judge: see if amplitude grows with level / compare which residual is cleaner.
5. **用途**: 季节调整/朴素预测/异常检测(盯残差)。
   Uses: seasonal adjustment/naive forecast/anomaly detection (watch residual).

### 下一节 / Next
**14.3 平滑法**——一类简单实用的预测方法: 用过去值的(加权)平均来平滑和外推。从移动平均、指数加权平均(EWMA), 到能同时处理趋势和季节的 **Holt-Winters 三重指数平滑**。
**14.3 Smoothing** — a simple, practical forecasting family: smooth and extrapolate via (weighted) averages of past values. From moving average, EWMA, to **Holt-Winters triple exponential smoothing** that handles both trend and seasonality.
